# 30-Day Readmission Modeling

This notebook will be the narrative layer for modeling, following `../docs/protocol.md`. Cohort SQL is already implemented in `../sql/`, with a final cohort of 824 patients. Feature logic and reusable functions should live in `sql/` and `src/`; this notebook should show the flow clearly without hiding scientific decisions.

**Current scope:** read the existing analytical table, validate the cohort, and compare M0 with M1 on one held-out test set. M2, tuning, calibration plots, and SHAP remain pending.

## 1. Setup

Load the small set of packages and project paths needed for the analysis. Keep environment setup outside the notebook so the run stays reproducible.

Select **Healthcare Readmission (.venv)** in VS Code/Jupyter. Installation commands and pinned dependencies are documented in `../README.md` and `../requirements.txt`.

In [1]:
from importlib.metadata import version
from pathlib import Path
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)

In [2]:
# Support execution from the repository root or anywhere within final_project.
for directory in (Path.cwd(), *Path.cwd().parents):
    if (directory / "final_project" / "notebooks").is_dir():
        PROJECT_DIR = directory / "final_project"
        break
else:
    raise FileNotFoundError("Run this notebook from the repository or final_project.")

DB_PATH = PROJECT_DIR / "data" / "omop_minimal.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Analytical database not found: {DB_PATH}")

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "readmission_30d"
M0_FEATURES = ["age", "sex", "prior_inpatient_hospitalizations_180d"]
ADDITIONAL_FEATURES = [
    "distinct_diagnoses_180d",
    "distinct_medications_180d",
    "measurement_count_180d",
]
M1_FEATURES = M0_FEATURES + ADDITIONAL_FEATURES
NUMERIC_FEATURES = [feature for feature in M1_FEATURES if feature != "sex"]
COUNT_FEATURES = [feature for feature in NUMERIC_FEATURES if feature != "age"]

print(f"Python: {sys.version.split()[0]} | {sys.executable}")
print(f"Database: {DB_PATH.relative_to(PROJECT_DIR.parent)}")
display(
    pd.Series(
        {
            package: version(package)
            for package in ["duckdb", "numpy", "pandas", "scikit-learn"]
        },
        name="Version",
    ).to_frame()
)

Python: 3.13.5 | /home/luis-enrique/Desktop/Datascience_BIomedicas/Labs - Clase - Tareas/.venv/bin/python
Database: final_project/data/omop_minimal.duckdb


,Version
duckdb,1.5.6
numpy,2.5.4
pandas,3.0.6
scikit-learn,1.9.1


## 2. Load Analytical Dataset

Load the one-row-per-patient analytical dataset after feature and outcome construction from the implemented OMOP cohort. `index_admission` and `index_discharge` are distinct anchors: the predictor window is `[index_admission - 180 days, index_admission)`, and readmission is a new inpatient hospitalization beginning after `index_discharge` and within the following 30 days.

Open DuckDB in read-only mode. Preserve the stored rows and values; ordering by `person_id` makes the seeded split independent of physical table order. No cohort, outcome, or feature SQL is rerun here.

In [3]:
with duckdb.connect(str(DB_PATH), read_only=True) as connection:
    analytic_df = connection.execute(
        "SELECT * FROM results.analytic_dataset ORDER BY person_id"
    ).fetchdf()

print(f"Loaded {len(analytic_df):,} rows and {analytic_df.shape[1]} columns.")
display(analytic_df.head(3).T.rename_axis("Column"))

Loaded 824 rows and 11 columns.


,0,1,2
Column,,,
person_id,25,35,76
index_visit_id,14721,14598,2293
index_admission,2019-07-02 12:38:58,2019-05-07 23:19:32,1987-06-03 07:35:27
index_discharge,2019-07-03 12:38:58,2019-05-09 07:18:06,1987-06-05 15:34:53
age,56,61,51
sex,F,F,F
prior_inpatient_hospitalizations_180d,0,0,0
distinct_diagnoses_180d,4,2,3
distinct_medications_180d,6,0,0


## 3. Cohort Quality Checks

Check the basics before modeling: one row per patient, outcome frequency, required columns, and no obvious missingness problems. These checks protect the analysis from quiet cohort mistakes.

Stop if the stored dataset differs from the expected **824 patients and 85 events**. These are table-level checks; the temporal feature definitions remain in the existing SQL. No rows are dropped and no values are imputed.

In [4]:
required_columns = [
    "person_id",
    "index_visit_id",
    "index_admission",
    "index_discharge",
    *M1_FEATURES,
    TARGET,
]
missing_columns = sorted(set(required_columns) - set(analytic_df.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"

assert len(analytic_df) == 824, "Expected 824 patients."
assert analytic_df["person_id"].is_unique, "Each patient must have one row."
assert analytic_df["index_visit_id"].is_unique, "Index visits must be unique."
assert not analytic_df[required_columns].isna().any().any(), (
    "Unexpected missing values."
)
assert analytic_df[TARGET].isin([0, 1]).all(), "Outcome must be binary."
assert analytic_df[TARGET].sum() == 85, "Expected 85 readmission events."
assert np.isfinite(analytic_df[NUMERIC_FEATURES].to_numpy()).all(), (
    "Nonfinite predictors."
)
assert analytic_df["age"].ge(18).all(), "The cohort must contain adults only."
assert analytic_df["sex"].isin(["F", "M"]).all(), "Unexpected source sex category."
assert analytic_df[COUNT_FEATURES].ge(0).all().all(), "Counts must be nonnegative."
assert (analytic_df[COUNT_FEATURES] % 1 == 0).all().all(), "Counts must be integers."
assert (analytic_df["index_discharge"] >= analytic_df["index_admission"]).all(), (
    "Discharge cannot precede admission."
)

cohort_summary = pd.DataFrame(
    {
        "Patients": [len(analytic_df)],
        "Events": [int(analytic_df[TARGET].sum())],
        "Non-events": [int(analytic_df[TARGET].eq(0).sum())],
        "Event rate (%)": [100 * analytic_df[TARGET].mean()],
        "Missing required values": [
            int(analytic_df[required_columns].isna().sum().sum())
        ],
    },
    index=["Analytical cohort"],
)
display(cohort_summary.round(2))
print("All cohort checks passed.")

,Patients,Events,Non-events,Event rate (%),Missing required values
Analytical cohort,824,85,739,10.32,0


All cohort checks passed.


## 4. Feature Overview

Review the six planned predictors: age, sex, previous inpatient hospitalizations, distinct diagnoses, distinct medications, and clinical measurement count. All longitudinal predictors use the 180-day window and end strictly before `index_admission`; the index hospitalization contributes no predictor information. The goal is to understand the inputs, not to search for extra features.

The tables describe the fixed inputs only. M0 uses the first three predictors; M1 adds the three clinical history counts. Descriptive summaries do not fit transformations or select predictors.

In [5]:
numeric_summary = analytic_df[NUMERIC_FEATURES].describe().T
numeric_summary.index.name = "Feature"
display(numeric_summary.round(2))

sex_summary = (
    analytic_df["sex"].value_counts().sort_index().rename("Patients").to_frame()
)
sex_summary["Percent"] = 100 * sex_summary["Patients"] / len(analytic_df)
sex_summary.index.name = "Sex (source value)"
display(sex_summary.round(2))

,count,mean,std,min,25%,50%,75%,max
Feature,,,,,,,,
age,824.0,60.54,10.90,21.0,51.0,61.0,68.0,98.0
prior_inpatient_hospitalizations_180d,824.0,0.03,0.23,0.0,0.0,0.0,0.0,3.0
distinct_diagnoses_180d,824.0,3.73,2.78,0.0,2.0,3.0,5.0,14.0
distinct_medications_180d,824.0,1.91,2.77,0.0,0.0,1.0,3.0,19.0
measurement_count_180d,824.0,41.31,75.98,0.0,1.0,17.0,35.0,553.0


,Patients,Percent
Sex (source value),,
F,440,53.4
M,384,46.6


## 5. Train/Test Split

Create a reproducible train/test split after the cohort and features are fixed. The split should preserve the outcome balance as much as the data allow.

Use **80% training / 20% test**, stratified by `readmission_30d`, with **random_state=42**. Both models use exactly the same patients. Patient IDs label rows for overlap checks; IDs and timestamps are excluded from predictors. Scaling and encoding are fitted inside each pipeline on training data only.

In [6]:
modeling_df = analytic_df.set_index("person_id")
X = modeling_df[M1_FEATURES]
y = modeling_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

assert X_train.index.intersection(X_test.index).empty, "Patient overlap between sets."
assert len(X_train) + len(X_test) == len(analytic_df)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert y_train.nunique() == y_test.nunique() == 2, "Each set needs both outcomes."

split_summary = pd.DataFrame(
    {
        "Patients": [len(y_train), len(y_test)],
        "Events": [int(y_train.sum()), int(y_test.sum())],
        "Non-events": [int(y_train.eq(0).sum()), int(y_test.eq(0).sum())],
        "Event rate (%)": [100 * y_train.mean(), 100 * y_test.mean()],
    },
    index=["Train", "Test"],
)
display(split_summary.round(2))

,Patients,Events,Non-events,Event rate (%)
Train,659,68,591,10.32
Test,165,17,148,10.30


## 6. M0 - Logistic Regression: Age + Sex + Previous Hospitalizations

Fit the baseline logistic regression using age, sex, and previous hospitalizations. This model establishes prediction based on demographics and prior hospital use, against which the additional longitudinal predictors are compared.

Use standard scaling for numeric predictors and one-hot encoding for sex (drop the first category). Both models use the same fixed L2-regularized logistic regression (`C=1`, `solver="lbfgs"`, `max_iter=2000`, `tol=1e-8`), with no class weighting, resampling, or tuning. Each call below creates a separate preprocessing pipeline.

Evaluate test-set probabilities with AUROC, AUPRC (**average precision**, without trapezoidal interpolation), and Brier score. Higher AUROC/AUPRC and lower Brier are better.

In [7]:
def make_logistic_pipeline(features):
    """Create fresh preprocessing and logistic regression for the selected predictors."""
    numeric_features = [feature for feature in features if feature != "sex"]
    preprocessor = ColumnTransformer(
        [
            ("numeric", StandardScaler(), numeric_features),
            ("sex", OneHotEncoder(drop="first", handle_unknown="error"), ["sex"]),
        ]
    )
    classifier = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        tol=1e-8,
        class_weight=None,
        random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocessor", preprocessor), ("classifier", classifier)])


def evaluate_probabilities(y_true, probabilities):
    """Evaluate positive-class probabilities on the held-out patients."""
    return {
        "AUROC": roc_auc_score(y_true, probabilities),
        "AUPRC (AP)": average_precision_score(y_true, probabilities),
        "Brier": brier_score_loss(y_true, probabilities),
    }

In [8]:
model_m0 = make_logistic_pipeline(M0_FEATURES)
model_m0.fit(X_train[M0_FEATURES], y_train)
assert model_m0.named_steps["classifier"].n_iter_.max() < 2000, "M0 did not converge."

probabilities_m0 = model_m0.predict_proba(X_test[M0_FEATURES])[:, 1]
metrics_m0 = evaluate_probabilities(y_test, probabilities_m0)
display(pd.DataFrame([metrics_m0], index=["M0"]).round(4))

,AUROC,AUPRC (AP),Brier
M0,0.6862,0.2142,0.0873


## 7. M1 - Logistic Regression: Full Feature Set

Fit the logistic regression model with all six planned predictors. This is the direct comparison against M0 for the value of longitudinal history.

Add `distinct_diagnoses_180d`, `distinct_medications_180d`, and `measurement_count_180d`. Keep the split, preprocessing rules, estimator settings, and evaluation unchanged.

In [9]:
model_m1 = make_logistic_pipeline(M1_FEATURES)
model_m1.fit(X_train[M1_FEATURES], y_train)
assert model_m1.named_steps["classifier"].n_iter_.max() < 2000, "M1 did not converge."

probabilities_m1 = model_m1.predict_proba(X_test[M1_FEATURES])[:, 1]
metrics_m1 = evaluate_probabilities(y_test, probabilities_m1)
display(pd.DataFrame([metrics_m1], index=["M1"]).round(4))

,AUROC,AUPRC (AP),Brier
M1,0.6888,0.2615,0.0878


### M0 vs M1 — held-out test comparison

Compare the models on the same test patients. The last row is **M1 minus M0**: positive differences favor M1 for AUROC/AUPRC, while a negative difference favors M1 for Brier. The test event rate provides context for average precision; a constant probability equal to the training event rate provides a Brier reference.

In [10]:
comparison = pd.DataFrame([metrics_m0, metrics_m1], index=["M0", "M1"])
comparison.index.name = "Model"
comparison_with_delta = comparison.copy()
comparison_with_delta.loc["M1 - M0"] = comparison.loc["M1"] - comparison.loc["M0"]
display(comparison_with_delta.round(4))

constant_probabilities = np.full(len(y_test), y_train.mean())
reference_brier = brier_score_loss(y_test, constant_probabilities)
print(f"Test set: {len(y_test)} patients; {int(y_test.sum())} events.")
print(f"Test event rate (AP reference): {y_test.mean():.4f}")
print(f"Constant training-prevalence prediction: {y_train.mean():.4f}")
print(f"Brier for that constant prediction on test: {reference_brier:.4f}")

,AUROC,AUPRC (AP),Brier
Model,,,
M0,0.6862,0.2142,0.0873
M1,0.6888,0.2615,0.0878
M1 - M0,0.0026,0.0473,0.0005


Test set: 165 patients; 17 events.
Test event rate (AP reference): 0.1030
Constant training-prevalence prediction: 0.1032
Brier for that constant prediction on test: 0.0924


## 8. M2 - XGBoost

Fit the XGBoost model with the same six predictors. This keeps the comparison fair: the model changes, but the feature set does not expand.

**Pending.** Not implemented or executed in this stage.

## 9. Light Optuna Tuning

If tuning is used, keep it light and limited to M2. The purpose is to avoid a weak default comparison, not to turn the project into a hyperparameter search study.

**Pending.** No tuning is implemented or executed in this stage.

## 10. Evaluation and SHAP

Evaluate AUROC, AUPRC, Brier score, and calibration. Interpret M2 using global SHAP values, summarized with mean absolute SHAP values and a SHAP summary plot. SHAP explains model predictions, not causal effects.

**Current status:** M0/M1 AUROC, AUPRC (average precision), and Brier are reported above. Calibration plots, M2 evaluation, and SHAP remain pending.

## 11. Interpretation and Limitations

Interpret only real results once they exist. Any conclusion must stay limited by the synthetic data, lack of external validation, and dependence on the OMOP cancer concept set.

The comparison uses one held-out split with a small number of events. Metric differences are descriptive and do not establish statistical superiority or clinical usefulness. Brier measures probability error; it does not establish calibration by itself. No model or threshold is tuned on the test set.

**Observed results:** the test set contains 165 patients and 17 events. M1 has slightly higher AUROC (0.6888 vs 0.6862) and higher average precision (0.2615 vs 0.2142), but slightly worse Brier (0.0878 vs 0.0873). Thus, the additional history features improve ranking on this split without improving overall probability error. Both models have higher average precision than the test event rate (0.1030) and lower Brier than the constant training-prevalence reference (0.0924). With only 17 test events, these differences should not be treated as stable estimates.